In [0]:
%run ./../config/00_project_config

In [0]:
%run ./../setup/00_storage_configuration

In [0]:
categories_bronze_path = f"{BRONZE_PATH}/categories"
categories_silver_path = f"{SILVER_PATH}/categories"

In [0]:
df_categories_bronze = spark.read.format("delta") \
    .load(categories_bronze_path)

In [0]:
df_categories_bronze.printSchema()

In [0]:
%skip
display(df_categories_bronze.limit(10))

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

windowSpec = Window.partitionBy("category_id").orderBy(F.col("created_at").desc())

df_categories_silver = df_categories_bronze \
    .withColumn(
        "rn",
        F.row_number().over(windowSpec)
    ) \
    .filter(F.col("rn") == 1) \
    .drop("rn")

In [0]:
df_categories_silver = df_categories_silver \
    .withColumn(
        "category_name",
        F.initcap(F.col("category_name"))
    )

In [0]:
%skip
display(df_categories_silver)

In [0]:
%skip
df_categories_silver.write.format("delta") \
    .mode("append") \
    .save(categories_silver_path)

In [0]:
from delta.tables import DeltaTable

silver_table = DeltaTable.forPath(
    spark,
    categories_silver_path
)

silver_table.alias("target") \
.merge(
    df_categories_silver.alias("source"),
    "target.category_id = source.category_id"
) \
.whenMatchedUpdate(
    set = {
        "category_name": "source.category_name"
    }
) \
.whenNotMatchedInsertAll() \
.execute()

In [0]:
%skip
display(
    spark.read.format("delta") \
        .load(categories_silver_path)
)